In [1]:
%pip install -q --upgrade pip

# Install a ragas version that supports langchain-core >=0.3,.
# plus a compatible langchain-community pin to avoid the ChatVertexAI import error.
%pip install -q \
    "ragas>=0.2.15" \
    "langchain-google-genai>=2.0.0" \
    "langchain-community<0.4.2" \
    langchain_cohere \
    langchain_core \
    datasets pandas matplotlib seaborn

In [2]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset
from ragas import evaluate, EvaluationDataset 
from ragas.metrics import SummarizationScore, SemanticSimilarity, AnswerCorrectness
from ragas.dataset_schema import SingleTurnSample
from langchain_google_genai import (
    ChatGoogleGenerativeAI,
    GoogleGenerativeAIEmbeddings,
)
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_cohere import ChatCohere
from ragas.llms import LangchainLLMWrapper

/tmp/ipykernel_15299/1040900535.py:7: DeprecationWarning: Importing SummarizationScore from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import SummarizationScore
  from ragas.metrics import SummarizationScore, SemanticSimilarity, AnswerCorrectness
/tmp/ipykernel_15299/1040900535.py:7: DeprecationWarning: Importing SemanticSimilarity from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import SemanticSimilarity
  from ragas.metrics import SummarizationScore, SemanticSimilarity, AnswerCorrectness
/tmp/ipykernel_15299/1040900535.py:7: DeprecationWarning: Importing AnswerCorrectness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import AnswerCorrectness
  from ragas.metrics import Summari

In [3]:
from getpass import getpass

COHERE_API_KEY = getpass("Enter your Cohere API key: ")

In [4]:
GOOGLE_API_KEY = getpass("Enter your GOOGLE API key: ")

In [5]:
os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY

In [6]:
import google.generativeai as genai

genai.configure(api_key=GOOGLE_API_KEY)

/usr/local/lib/python3.13/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


In [7]:
from langchain_core.outputs import LLMResult

def cohere_is_finished_parser(response: LLMResult) -> bool:
    """
    RAGAS-compatible finished parser for Cohere models.
    Returns True when Cohere signals successful completion.
    """
    if not response.generations:
        return False
    gen = response.generations[0][0]
    # Cohere may expose finish_reason in generation_info
    if gen.generation_info:
        finish_reason = gen.generation_info.get("finish_reason", "")
        # Cohere uses 'COMPLETE'; some versions use 'end_turn'
        if finish_reason in ("COMPLETE", "end_turn", "stop"):
            return True
    # Fallback: check message metadata (for newer langchain-cohere)
    if hasattr(gen, "message") and gen.message:
        stop_reason = gen.message.response_metadata.get("stop_reason", "")
        if stop_reason in ("COMPLETE", "end_turn", "stop"):
            return True
    return False

In [8]:
cohere_llm = ChatCohere(
    cohere_api_key=COHERE_API_KEY,
    model="command-a-03-2025",
    temperature=0,
    max_tokens=8192,          # RAGAS rubrics can be verbose
)

# Wrap for RAGAS (required for evaluate() / metric-level llm param
judge_llm = LangchainLLMWrapper(cohere_llm, is_finished_parser=cohere_is_finished_parser,)

print(f"✅ Cohere Command A judge ready (model: {cohere_llm.model})")

✅ Cohere Command A judge ready (model: command-a-03-2025)


/tmp/ipykernel_15299/1229937477.py:9: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  judge_llm = LangchainLLMWrapper(cohere_llm, is_finished_parser=cohere_is_finished_parser,)


In [9]:
# llm = ChatGoogleGenerativeAI(
#     model="gemini-3.5-flash-lite",
#     response_mime_type="application/json",  # forces raw JSON, no fences
#     google_api_key="GOOGLE_API_KEY",
# )

embeddings = LangchainEmbeddingsWrapper(
    GoogleGenerativeAIEmbeddings(
        model="gemini-embedding-001",  # or "models/embedding-001"
        google_api_key="GOOGLE_API_KEY",
    )
)

/tmp/ipykernel_15299/2791003649.py:7: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  embeddings = LangchainEmbeddingsWrapper(


In [10]:
ds = load_dataset("pameydorke/redred-gemma-4-E2B-it-lora-summaries", split="train")
df = ds.to_pandas()

# Show the columns and a sample row
print(df.columns.tolist())
df.head(2)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


['user_input', 'reference', 'base_response', 'ft_response']


,user_input,reference,base_response,ft_response
0,Original Post: Help with Small living room Use...,The OP asked about general design suggestions ...,"The original poster, moving into a 1920s craft...",The user is asking for advice on how to arrang...
1,Original Post: What to do with this half a cyl...,The OP wanted advice on what to do with the wo...,The original poster asked for ideas on what to...,"A user on the ""designmyroom"" subreddit is aski..."


In [11]:
def build_samples(row, response_col):
    """Create a SingleTurnSample for Ragas from a dataframe row."""
    return SingleTurnSample(
        user_input=row["user_input"],          # normalized Reddit thread text
        response=row[response_col],            # model summary (base or ft)
        reference=row["reference"],            # human‑made summary
        # SummarizationScore needs the original context to extract keyphrases
        reference_contexts=[row["user_input"]],
    )

# Build sample lists for the base model and the fine‑tuned model
base_samples = [build_samples(row, "base_response") for _, row in df.iterrows()]
ft_samples   = [build_samples(row, "ft_response")   for _, row in df.iterrows()]

base_dataset = EvaluationDataset(samples=base_samples)
ft_dataset = EvaluationDataset(samples=ft_samples)

In [12]:
# Define the metrics we want to compute
metrics = [
    SummarizationScore(llm=judge_llm, coeff=0.5),  # 0.5 balances QA vs. conciseness
    SemanticSimilarity(embeddings=embeddings),
    AnswerCorrectness(llm=judge_llm, embeddings=embeddings),
]

# Evaluate the base model summaries
base_result = evaluate(
    dataset=base_dataset,
    metrics=metrics,
    llm=judge_llm,
    embeddings=embeddings,
)

# Evaluate the fine‑tuned model summaries
ft_result = evaluate(
    dataset=ft_dataset,
    metrics=metrics,
    llm=judge_llm,
    embeddings=embeddings,
)

# Convert results to dataframes for easy inspection
base_df = base_result.to_pandas()
ft_df   = ft_result.to_pandas()

print("Base model scores:")
print(base_df.mean(numeric_only=True))
print("\nFine‑tuned model scores:")
print(ft_df.mean(numeric_only=True))

Evaluating:   0%|          | 0/66 [00:00<?, ?it/s]

ERROR:ragas.executor:Exception raised in Job[10]: GoogleGenerativeAIError(Error embedding content (INVALID_ARGUMENT): 400 INVALID_ARGUMENT. {'error': {'code': 400, 'message': 'API key not valid. Please pass a valid API key.', 'status': 'INVALID_ARGUMENT', 'details': [{'@type': 'type.googleapis.com/google.rpc.ErrorInfo', 'reason': 'API_KEY_INVALID', 'domain': 'googleapis.com', 'metadata': {'service': 'generativelanguage.googleapis.com'}}, {'@type': 'type.googleapis.com/google.rpc.LocalizedMessage', 'locale': 'en-US', 'message': 'API key not valid. Please pass a valid API key.'}]}})
ERROR:ragas.executor:Exception raised in Job[7]: GoogleGenerativeAIError(Error embedding content (INVALID_ARGUMENT): 400 INVALID_ARGUMENT. {'error': {'code': 400, 'message': 'API key not valid. Please pass a valid API key.', 'status': 'INVALID_ARGUMENT', 'details': [{'@type': 'type.googleapis.com/google.rpc.ErrorInfo', 'reason': 'API_KEY_INVALID', 'domain': 'googleapis.com', 'metadata': {'service': 'generativ

KeyboardInterrupt: 

ERROR:ragas.executor:Exception raised in Job[13]: GoogleGenerativeAIError(Error embedding content (INVALID_ARGUMENT): 400 INVALID_ARGUMENT. {'error': {'code': 400, 'message': 'API key not valid. Please pass a valid API key.', 'status': 'INVALID_ARGUMENT', 'details': [{'@type': 'type.googleapis.com/google.rpc.ErrorInfo', 'reason': 'API_KEY_INVALID', 'domain': 'googleapis.com', 'metadata': {'service': 'generativelanguage.googleapis.com'}}, {'@type': 'type.googleapis.com/google.rpc.LocalizedMessage', 'locale': 'en-US', 'message': 'API key not valid. Please pass a valid API key.'}]}})
